# Evaluation-Driven Development

[Read the cookbook page](https://docs.futureagi.com/docs/cookbook/quickstart/eval-driven-dev).

## Install

Start in a fresh Python 3.12 kernel. Use environment variables or enter the three keys at the prompts.

In [ ]:
import subprocess
import sys

subprocess.run([sys.executable, "-m", "pip", "install", "--quiet",
                'agent-learning-kit==0.3.7', 'openai==2.54.0'], check=True)

In [ ]:
import os
from getpass import getpass

for name in ('FI_API_KEY', 'FI_SECRET_KEY', 'OPENAI_API_KEY'):
    if not os.environ.get(name):
        value = getpass(f"{name}: ")
        if not value.strip():
            raise ValueError(f"{name} is required.")
        os.environ[name] = value.strip()
        del value

## Define the eval suite

In [ ]:
TEST_CASES = [
    {
        "input": "What is your return window for electronics?",
        "context": (
            "Electronics may be returned within 30 days of purchase with original "
            "packaging and proof of purchase. Items must be in unused condition."
        ),
    },
    {
        "input": "My order arrived damaged. What should I do?",
        "context": (
            "Customers who receive damaged items should photograph the damage and "
            "contact support within 48 hours. A replacement or full refund will be "
            "issued after review."
        ),
    },
    {
        "input": "Can I return a sale item for a full refund?",
        "context": (
            "Sale items are eligible for exchange only. Full refunds are not available "
            "on sale purchases. Store credit may be offered at management discretion."
        ),
    },
    {
        "input": "How long does standard shipping take?",
        "context": (
            "Standard shipping takes 5-7 business days within the continental US. "
            "Expedited options (2-day and overnight) are available at checkout."
        ),
    },
    {
        "input": "Do you price-match competitors?",
        "context": (
            "We offer a price-match guarantee for identical items sold by authorized "
            "retailers. The match must be requested at the time of purchase. "
            "Marketplace sellers and auction sites are excluded."
        ),
    },
]

## Write the scoring function

Faithfulness runs locally. Toxicity uses the hosted Turing model.

In [ ]:
import json
from pathlib import Path
from openai import OpenAI
from fi.evals import evaluate

openai_client = OpenAI()


def score_prompt(prompt_template: str, test_cases: list) -> dict:
    if not test_cases:
        raise ValueError("Supply at least one test case.")
    faithfulness_passes = 0
    toxicity_passes = 0
    per_case = []

    for case in test_cases:
        system_prompt = prompt_template.format(context=case["context"])

        response = openai_client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user",   "content": case["input"]},
            ],
        )
        output = response.choices[0].message.content

        if not output or not output.strip():
            raise RuntimeError("No reply returned for " + case["input"])

        # faithfulness: local metric, no model= argument
        faith_result = evaluate(
            "faithfulness",
            output=output,
            context=case["context"],
        )

        # toxicity: Turing metric, model= is required
        tox_result = evaluate(
            "toxicity",
            output=output,
            model="turing_small",
        )

        for result in (faith_result, tox_result):
            if result.status != "completed" or result.score is None or result.passed is None:
                raise RuntimeError(f"{result.eval_name} did not score: {result.error or result.reason}")
        if faith_result.passed:
            faithfulness_passes += 1
        if tox_result.passed:
            toxicity_passes += 1

        per_case.append({
            "input":               case["input"],
            "output":              output,
            "faithfulness_score":  faith_result.score,
            "faithfulness_pass":   faith_result.passed,
            "faithfulness_reason": faith_result.reason,
            "toxicity_eval_id":     tox_result.metadata["eval_id"],
            "toxicity_score":      tox_result.score,
            "toxicity_pass":       tox_result.passed,
            "toxicity_reason":     tox_result.reason,
        })

    n = len(test_cases)
    faith_rate = faithfulness_passes / n
    tox_rate   = toxicity_passes / n

    return {
        "faithfulness": faith_rate,
        "toxicity":     tox_rate,
        "composite":    (faith_rate + tox_rate) / 2,
        "per_case":     per_case,
    }

In [ ]:
smoke = evaluate(
    "faithfulness",
    output="Electronics can be returned within 30 days with proof of purchase.",
    context=TEST_CASES[0]["context"],
)
print(smoke.score, smoke.reason)
# Compare the score with the reason; the local check is an approximation.

## Score a baseline and a revised prompt

In [ ]:
BASELINE_PROMPT = """\
You are a customer support agent.
Answer the customer's question using the information below.

Context:
{context}
"""

REVISED_PROMPT = """\
You are a friendly and professional customer support agent for Ridgeline Outfitters.

INSTRUCTIONS:
1. Answer ONLY using the information provided in the Context section below.
2. Do NOT add policies, timeframes, or details that are not stated in the Context.
3. If the Context does not contain enough information to fully answer the question,
   say so clearly and offer to escalate to the support team.
4. Prefer quoting the relevant policy sentence. Keep the answer under three sentences.

Context:
{context}
"""


def print_results(label: str, results: dict):
    print(f"\n{'='*40}")
    print(f"  {label}")
    print(f"{'='*40}")
    print(f"{'Metric':<16} {'Pass rate':>10}")
    print("-" * 28)
    print(f"{'faithfulness':<16} {results['faithfulness']:>9.0%}")
    print(f"{'toxicity':<16} {results['toxicity']:>9.0%}")
    print(f"{'composite':<16} {results['composite']:>9.0%}")

    for i, case in enumerate(results["per_case"], 1):
        faith = "PASS" if case["faithfulness_pass"] else "FAIL"
        tox   = "PASS" if case["toxicity_pass"] else "FAIL"
        print(f"  [{i}] {case['input'][:50]:<52} faith={faith}  tox={tox}")


# Run both
baseline = score_prompt(BASELINE_PROMPT, TEST_CASES)
revised  = score_prompt(REVISED_PROMPT, TEST_CASES)

print_results("BASELINE", baseline)
print_results("REVISED", revised)
for name, report in [("baseline", baseline), ("revised", revised)]:
    Path(f"{name}.json").write_text(json.dumps(report, indent=2))

# Show delta
print(f"\n--- Comparison ---")
print(f"faithfulness: {baseline['faithfulness']:.0%} → {revised['faithfulness']:.0%}")
print(f"toxicity:     {baseline['toxicity']:.0%} → {revised['toxicity']:.0%}")
print(f"composite:    {baseline['composite']:.0%} → {revised['composite']:.0%}")

## Inspect the remaining failure

If all cases pass, the loop prints nothing.

In [ ]:
for case in revised["per_case"]:
    if not case["faithfulness_pass"]:
        print(f"Input:  {case['input']}")
        print(f"Output: {case['output']}")
        print(f"Score:  {case['faithfulness_score']:.2f}")
        print(f"Reason: {case['faithfulness_reason']}")

## Gate promotion on eval thresholds

Write the CI script, then run it on the same revised report. Exit 1 with `GATE FAILED` is an evaluation outcome, not an execution error.

In [ ]:
%%writefile gate.py
import json
import sys
from pathlib import Path

thresholds = {"faithfulness": 0.75, "toxicity": 0.80}
results = json.loads(Path(sys.argv[1]).read_text())
failed = []
for metric, threshold in thresholds.items():
    rate = results[metric]
    if not isinstance(rate, (int, float)) or not 0 <= rate <= 1:
        raise ValueError(f"Invalid pass rate for {metric}")
    print(f"{metric}: {rate:.0%}  (threshold: {threshold:.0%})")
    if rate < threshold:
        failed.append(metric)
if failed:
    print("GATE FAILED: " + ", ".join(failed))
    sys.exit(1)
print("Quality gate passed.")

In [ ]:
gate = subprocess.run([sys.executable, "gate.py", "revised.json"], capture_output=True, text=True)
print(gate.stdout, end="")
if not (gate.returncode == 1 and "GATE FAILED:" in gate.stdout):
    gate.check_returncode()